<a href="https://colab.research.google.com/github/yokunerukosbelt/mask/blob/main/Copy_of_Hypothalamus_mask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q nibabel simpleitk matplotlib numpy ipywidgets
from google.colab import files
import nibabel as nib, numpy as np, matplotlib.pyplot as plt
import SimpleITK as sitk
from ipywidgets import interact, IntSlider, fixed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 109.7 MB/s eta 0:00:00


In [3]:
print("Upload atlas_T2.nii.gz, atlas_labels.nii.gz, and your MRI...")
uploaded = files.upload()

Upload atlas_T2.nii.gz, atlas_labels.nii.gz, and your MRI...


Saving sub-001_ses-1_acq-RARE_T2w.nii to sub-001_ses-1_acq-RARE_T2w (1).nii
Saving WAXHOLM_SPACE_ATLAS_OF_THE_SPRAGUE_DAWLEY_RAT_BRAIN_V4.label.nii to WAXHOLM_SPACE_ATLAS_OF_THE_SPRAGUE_DAWLEY_RAT_BRAIN_V4.label.nii
Saving WAXHOLM_SPACE_OF_THE_SPRAGUE_DAWLEY_V1_01.nii to WAXHOLM_SPACE_OF_THE_SPRAGUE_DAWLEY_V1_01.nii


In [4]:
lab_img = nib.load('WAXHOLM_SPACE_ATLAS_OF_THE_SPRAGUE_DAWLEY_RAT_BRAIN_V4.label.nii'); lab = lab_img.get_fdata().astype(int)
ids, counts = np.unique(lab, return_counts=True)
print("Found label IDs:", ids[:50], " ... total:", len(ids))
# If you have a label table, map names→IDs; otherwise set the ID manually below.

Found label IDs: [ 0  1  3  4  5  6  7 10 32 33 34 35 36 37 38 40 41 42 43 45 46 47 48 50
 51 52 53 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 71 73 74 75 76 77
 78 79]  ... total: 215


In [5]:
HYP_ID = 48  # <-- CHANGE if needed based on your atlas
assert HYP_ID in ids, f"Hypothalamus ID {HYP_ID} not found in label map!"

In [6]:
hypo_mask = (lab == HYP_ID).astype(np.uint8)
hypo_atlas_img = nib.Nifti1Image(hypo_mask, affine=lab_img.affine, header=lab_img.header)
nib.save(hypo_atlas_img, "/content/hypothalamus_mask_atlas_space.nii.gz")
print("Saved: /content/hypothalamus_mask_atlas_space.nii.gz")
files.download("/content/hypothalamus_mask_atlas_space.nii.gz")

Saved: /content/hypothalamus_mask_atlas_space.nii.gz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
# Read with SimpleITK
atlas_t2_sitk = sitk.ReadImage('/content/WAXHOLM_SPACE_OF_THE_SPRAGUE_DAWLEY_V1_01.nii', sitk.sitkFloat32)
mri_sitk      = sitk.ReadImage('/content/sub-001_ses-1_acq-RARE_T2w.nii',      sitk.sitkFloat32)

# Initial alignment (centered)
initial_tx = sitk.CenteredTransformInitializer(
    mri_sitk, atlas_t2_sitk, sitk.Euler3DTransform(),
    sitk.CenteredTransformInitializerFilter.GEOMETRY)

# Set up registration (Mattes MI + gradient descent)
reg = sitk.ImageRegistrationMethod()
reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
reg.SetMetricSamplingStrategy(reg.RANDOM)
reg.SetMetricSamplingPercentage(0.2)
reg.SetInterpolator(sitk.sitkLinear)
reg.SetOptimizerAsRegularStepGradientDescent(learningRate=2.0,
                                            minStep=1e-4,
                                            numberOfIterations=200,
                                            relaxationFactor=0.5)
reg.SetOptimizerScalesFromPhysicalShift()
reg.SetInitialTransform(initial_tx, inPlace=False)

final_rigid = reg.Execute(mri_sitk, atlas_t2_sitk)
print("Rigid registration done. Metric:", reg.GetMetricValue())

# (Optional) Affine refinement
do_affine = False  # set True if you need finer fit
if do_affine:
    affine = sitk.AffineTransform(final_rigid)
    reg2 = sitk.ImageRegistrationMethod()
    reg2.SetMetricAsMattesMutualInformation(50)
    reg2.SetMetricSamplingStrategy(reg2.RANDOM)
    reg2.SetMetricSamplingPercentage(0.2)
    reg2.SetInterpolator(sitk.sitkLinear)
    reg2.SetOptimizerAsRegularStepGradientDescent(1.0, 1e-4, 200, 0.5)
    reg2.SetInitialTransform(affine, inPlace=False)
    final_tx = reg2.Execute(mri_sitk, atlas_t2_sitk)
    print("Affine refinement done. Metric:", reg2.GetMetricValue())
else:
    final_tx = final_rigid

Rigid registration done. Metric: -0.4603160158076683


In [8]:
# Read mask in atlas space
hypo_atlas_sitk = sitk.ReadImage("/content/hypothalamus_mask_atlas_space.nii.gz", sitk.sitkUInt8)

# Resample to MRI space using nearest-neighbor (preserve labels)
resampled_hypo = sitk.Resample(
    hypo_atlas_sitk,
    mri_sitk,
    final_tx,
    sitk.sitkNearestNeighbor,
    0,  # default pixel value
    sitk.sitkUInt8)

out_mask_path = "/content/hypothalamus_mask_in_MRI_space.nii.gz"
sitk.WriteImage(resampled_hypo, out_mask_path)
print("Saved:", out_mask_path)
files.download(out_mask_path)

Saved: /content/hypothalamus_mask_in_MRI_space.nii.gz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# Load with nibabel for plotting
mri_path = '/content/sub-001_ses-1_acq-RARE_T2w.nii' # Define mri_path
mri_np  = nib.load(mri_path).get_fdata()
mask_np = nib.load(out_mask_path).get_fdata().astype(bool)

# Normalize MRI for display
mri_norm = (mri_np - np.min(mri_np)) / (np.max(mri_np) - np.min(mri_np) + 1e-8)

def show_slice(k=0, axis='axial'):
    if axis=='axial':
        img = mri_norm[:, :, k]; m = mask_np[:, :, k]
    elif axis=='coronal':
        img = mri_norm[:, k, :]; m = mask_np[:, k, :]
    else:  # sagittal
        img = mri_norm[k, :, :]; m = mask_np[k, :, :]

    plt.figure(figsize=(6,6))
    plt.imshow(np.rot90(img), cmap='gray')
    plt.imshow(np.rot90(np.where(m,1,0)), cmap='Greens', alpha=0.45)
    plt.title(f'{axis.capitalize()} slice {k}')
    plt.axis('off'); plt.show()

# Choose a view and interact
axis = 'axial'   # 'axial' | 'coronal' | 'sagittal'
interact(show_slice, k=IntSlider(min=0, max=mri_np.shape[2 if axis=='axial' else 1 if axis=='coronal' else 0]-1, step=1, value=(mri_np.shape[2 if axis=='axial' else 1 if axis=='coronal' else 0]//2)), axis=fixed(axis))

interactive(children=(IntSlider(value=6, description='k', max=11), Output()), _dom_classes=('widget-interact',…

<function __main__.show_slice(k=0, axis='axial')>

In [10]:
def save_slices(mri_np, mask_np, mri_norm, axis='axial', start_slice=None, end_slice=None):
    num_slices = mri_np.shape[2 if axis=='axial' else 1 if axis=='coronal' else 0]
    output_dir = f"/content/slices_{axis}"
    !mkdir -p {output_dir}

    if start_slice is None:
        start_slice = 0
    if end_slice is None:
        end_slice = num_slices

    for k in range(start_slice, end_slice):
        if axis=='axial':
            img = mri_norm[:, :, k]; m = mask_np[:, :, k]
        elif axis=='coronal':
            img = mri_norm[:, k, :]; m = mask_np[:, k, :]
        else:  # sagittal
            img = mri_norm[k, :, :]; m = mask_np[k, :, :]

        plt.figure(figsize=(6,6))
        plt.imshow(np.rot90(img), cmap='gray')
        plt.imshow(np.rot90(np.where(m,1,0)), cmap='Greens', alpha=0.45)
        plt.title(f'{axis.capitalize()} slice {k}')
        plt.axis('off')
        jpeg_out = f"{output_dir}/{axis}_slice_{k:03d}.jpg"
        plt.savefig(jpeg_out, bbox_inches='tight', pad_inches=0, dpi=100)
        plt.close()
        print(f"Saved: {jpeg_out}")

# Save axial slices 0-11
save_slices(mri_np, mask_np, mri_norm, axis='axial', start_slice=0, end_slice=12)

# Save coronal slices
# save_slices(mri_np, mask_np, mri_norm, axis='coronal')

# Save sagittal slices
# save_slices(mri_np, mask_np, mri_norm, axis='sagittal')

print("Finished saving specified axial slices.")

Saved: /content/slices_axial/axial_slice_000.jpg
Saved: /content/slices_axial/axial_slice_001.jpg
Saved: /content/slices_axial/axial_slice_002.jpg
Saved: /content/slices_axial/axial_slice_003.jpg
Saved: /content/slices_axial/axial_slice_004.jpg
Saved: /content/slices_axial/axial_slice_005.jpg
Saved: /content/slices_axial/axial_slice_006.jpg
Saved: /content/slices_axial/axial_slice_007.jpg
Saved: /content/slices_axial/axial_slice_008.jpg
Saved: /content/slices_axial/axial_slice_009.jpg
Saved: /content/slices_axial/axial_slice_010.jpg
Saved: /content/slices_axial/axial_slice_011.jpg
Finished saving specified axial slices.


You can view the saved JPEG files in the `/content/slices_axial`, `/content/slices_coronal`, and `/content/slices_sagittal` directories in the Colab file browser.